In [3]:
# Installing needed package (triton)
! pip install triton
! pip install torch


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.5/915.5 MB 914.1 kB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 3.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 32.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 4.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 58.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 3.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 20.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 97.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 30.4 MB/s eta 0:00:00
     ━━━━

In [17]:
! pip install numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 79.9 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:
# Importing needed libraries
import torch
import torch.nn.functional as F
import triton
import triton.language as tl
import time

/home/jason/CUDA1/.venv/lib/python3.10/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [26]:
# Golbal Variables
DEVICE = "cuda"
C_OUT = 64
C_IN = 3
H = 1024
W = 1024
FH = 3
FW = 3
BLOCK_SIZE = 16
PADDED_SIZE = 32 
F_SIZE = 16

In [98]:
# Making torch tensors
tensor_I = torch.rand(1, C_IN, H, W, device=DEVICE, dtype=torch.float64) # Input, assuming that batch_size is one
tensor_F = torch.rand(C_OUT, C_IN, FH, FW , device=DEVICE, dtype=torch.float64) # Weights

In [99]:
# This is the result from Convolutional Layer provided by Torch
# Use this for correctness check
golden_out = F.conv2d(tensor_I, tensor_F, padding=1)
print(golden_out.shape) # (1, C_OUT, OUT_H, OUT_W)

torch.Size([1, 64, 1024, 1024])


In [ ]:
@triton.jit
def my_triton_kernel(
  i0_ptr, f_ptr, o_ptr,
  C_OUT,  
  C_IN,
  H,
  W,
  FH: tl.constexpr,
  FW: tl.constexpr,
  BLOCK_SIZE: tl.constexpr,
  PADDED_SIZE: tl.constexpr,
  F_SIZE: tl.constexpr,
  ):
  """
  This is a triton kernel that does Conv assuming that padding=1, stride=1
  You should 1) load the values fro input and kernel, 2) does computation, 3) store the result
  """
  pid_k = tl.program_id(2)
  block_row = tl.program_id(1)
  block_col = tl.program_id(0)

  offs_row = block_row * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
  offs_col = block_col * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)

  i0_offs_row = block_row * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
  i0_offs_col = block_col * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
  acc = tl.zeros((BLOCK_SIZE, BLOCK_SIZE), dtype=tl.float64)
  
  for c in range (C_IN):
    for i in range (FH):
      for j in range (FW):

        f_val = tl.load(f_ptr + pid_k * C_IN * FH * FW 
          + c * FH * FW 
          + (i) * FW + (j))

        i0_ptrs = i0_ptr + c * (H+2)*(W+2) + (i0_offs_row+i)[:, None] * (W+2) + (i0_offs_col+j)[None, :]

        i0 = tl.load(i0_ptrs)

        acc += i0 * f_val

  o_ptrs = o_ptr + pid_k * H * W + offs_row[:, None] * W + offs_col[None, :]
  tl.store(o_ptrs, acc)
  pass

def my_conv2d(input, kernel):
  """
  This function is a wrapper function that preprocess the inputs and call the kernel
  input: torch.tensor (1, C_IN, H, W)
  kernel: torch.tensor (C_OUT, C_IN, FH, FW)
  """
  # Initializing some variables
  i0_padded = F.pad(input, (1, 1, 1, 1))

  # Calculate output dimension & Allocate output tensor
  output_tensor = torch.zeros(1, C_OUT, H, W, dtype=torch.float64, device='cuda')

  # Define grid
  grid = (
    triton.cdiv(W, BLOCK_SIZE),
    triton.cdiv(H, BLOCK_SIZE),
    C_OUT
  )
  
  # Call the triton kernel (my_triton_kernel) and measure execution time

  start = torch.cuda.Event(enable_timing=True)
  end   = torch.cuda.Event(enable_timing=True)

  start.record()
  my_triton_kernel[grid](
    i0_padded, kernel, output_tensor,
    C_OUT,  
    C_IN,
    H,
    W,
    FH,
    FW,
    BLOCK_SIZE = BLOCK_SIZE,
    PADDED_SIZE = PADDED_SIZE,
    F_SIZE = F_SIZE,
  )

  end.record()
  torch.cuda.synchronize()

  # Return output (output should include execution time)
  return output_tensor, start.elapsed_time(end)

In [123]:
# Testing

# Comparing the result from my_conv2d and Conv from torch
my_output, execution_time = my_conv2d(tensor_I, tensor_F)

torch.testing.assert_close(golden_out, my_output) # Assert statement should be passed
# Printing the execution time
print(f"Execution Time for triton kernel: {execution_time:.3f} ms")

checksum:455975131.1060321
Execution Time for triton kernel: 8.008 ms
